# RAG Knowledge Base Initialization

This notebook builds your RAG system knowledge base:
1. **Load** PDFs from `data/raw_papers/`
2. **Chunk** them into manageable pieces
3. **Store** chunks in `data/chunks.jsonl` (ready for embeddings)
4. **Query** and verify the stored knowledge

Run cells sequentially from top to bottom.

## 0: Setup & Load Configuration

In [1]:
import logging
from pathlib import Path
from datetime import datetime

from backend.core.config import settings
from backend.services.paper_processor import PaperProcessor
from backend.services.document_loader import DocumentLoader
from backend.services.chunk_store import ChunkStore
from backend.services.paper_registry import PaperRegistry
from backend.services.query_transformer import QueryTransformer

logging.basicConfig(
    level=settings.log_level,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Suppress verbose logs from HuggingFace Hub
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

print("✓ Imports successful (kernel restarted)")
print(f"\n--- Configuration ---")
print(f"Data directory:    {settings.data_dir}")
print(f"Raw papers:        {settings.raw_papers_dir}")
print(f"Chunks file:       {settings.chunks_file}")
print(f"Manifest:          {settings.manifest_path}")
print(f"\n--- Chunking Config ---")
print(f"Chunk size:        {settings.chunk_size}")
print(f"Chunk overlap:     {settings.chunk_overlap}")
print(f"Log level:         {settings.log_level}")

d:\Programming\projects\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Imports successful (kernel restarted)

--- Configuration ---
Data directory:    d:\Programming\projects\RAG\data
Raw papers:        d:\Programming\projects\RAG\data\raw_papers
Chunks file:       d:\Programming\projects\RAG\data\chunks.jsonl
Manifest:          d:\Programming\projects\RAG\data\papers_manifest.json

--- Chunking Config ---
Chunk size:        1000
Chunk overlap:     200
Log level:         INFO


## 1: Check Prerequisites


In [2]:
pdf_files = list(settings.raw_papers_dir.glob("*.pdf"))

print(f"Checking {settings.raw_papers_dir}...")
print(f"\nFound {len(pdf_files)} PDF files:")

if not pdf_files:
    print("  ❌ NO PDFs FOUND!")
    print(f"\nAdd PDF files to {settings.raw_papers_dir} before continuing.")
else:
    for i, pdf in enumerate(pdf_files, 1):
        size_mb = pdf.stat().st_size / (1024*1024)
        print(f"  {i}. {pdf.name:<40} ({size_mb:.2f} MB)")
    
    print(f"\n✓ Ready to process {len(pdf_files)} PDFs")

Checking d:\Programming\projects\RAG\data\raw_papers...

Found 18 PDF files:
  1. 1906_MC_Layer_Normalization_fo.pdf       (0.65 MB)
  2. accurate_uncertanities_for_dl_using_calibrated_regression.pdf (2.71 MB)
  3. Active_Learning_Framework_for_Improving_Knowledge_Graph_Accuracy.pdf (2.88 MB)
  4. automated_construction_of_theme_specific_kg.pdf (1.29 MB)
  5. dibowski-full-traceability-and-provenance-for-knowledge-graphs.pdf (0.47 MB)
  6. DOC-20251031-WA0004.pdf                  (0.20 MB)
  7. DOC-20251031-WA0005.pdf                  (0.44 MB)
  8. DOC-20251031-WA0006.pdf                  (1.06 MB)
  9. DOC-20251031-WA0007.pdf                  (1.81 MB)
  10. DOC-20251031-WA0008.pdf                  (1.15 MB)
  11. DOC-20251031-WA0009.pdf                  (0.38 MB)
  12. DOC-20251031-WA0010.pdf                  (1.31 MB)
  13. DOC-20251031-WA0011.pdf                  (0.87 MB)
  14. DOC-20251031-WA0012.pdf                  (0.27 MB)
  15. DOC-20251031-WA0013.pdf                  (0.69

## 2: Check Current Status


In [3]:
registry = PaperRegistry(settings.manifest_path)
store = ChunkStore(settings.chunks_file)

print("=" * 60)
print("CURRENT STATUS")
print("=" * 60)

status = registry.get_status()
print(f"\n--- Registry (papers_manifest.json) ---")
print(f"Papers tracked:    {status['total_papers']}")
print(f"Total chunks:      {status['total_chunks']}")

store_stats = store.get_stats()
print(f"\n--- Chunk Store (chunks.jsonl) ---")
print(f"Chunks stored:     {store_stats['total_chunks']}")
print(f"Sources (papers):  {store_stats['total_sources']}")

if status['total_papers'] > 0:
    print(f"\n--- Tracked Papers ---")
    for paper in status['papers']:
        print(f"  • {paper['filename']:<40} ({paper['num_chunks']} chunks)")


2026-03-23 19:30:01,447 - backend.services.paper_registry - INFO - Loaded manifest: 18 papers


CURRENT STATUS

--- Registry (papers_manifest.json) ---
Papers tracked:    18
Total chunks:      1387

--- Chunk Store (chunks.jsonl) ---
Chunks stored:     1387
Sources (papers):  18

--- Tracked Papers ---
  • 1906_MC_Layer_Normalization_fo.pdf       (82 chunks)
  • accurate_uncertanities_for_dl_using_calibrated_regression.pdf (55 chunks)
  • Active_Learning_Framework_for_Improving_Knowledge_Graph_Accuracy.pdf (87 chunks)
  • automated_construction_of_theme_specific_kg.pdf (77 chunks)
  • dibowski-full-traceability-and-provenance-for-knowledge-graphs.pdf (63 chunks)
  • DOC-20251031-WA0004.pdf                  (28 chunks)
  • DOC-20251031-WA0005.pdf                  (90 chunks)
  • DOC-20251031-WA0006.pdf                  (69 chunks)
  • DOC-20251031-WA0007.pdf                  (98 chunks)
  • DOC-20251031-WA0008.pdf                  (164 chunks)
  • DOC-20251031-WA0009.pdf                  (53 chunks)
  • DOC-20251031-WA0010.pdf                  (69 chunks)
  • DOC-20251031-WA0011.p

## 3: Process Papers (Full Pipeline)

This is the main operation:
- Detects new PDFs (not in manifest)
- Detects updated PDFs (file hash changed)
- For each: Load → Chunk → Store → Register

Idempotent - safe to run multiple times.

In [4]:
processor = PaperProcessor(
    papers_dir=settings.raw_papers_dir,
    manifest_path=settings.manifest_path,
    chunks_file=settings.chunks_file,
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
)

print(f"\n{'='*60}")
print("PROCESSING PAPERS")
print(f"{'='*60}\n")

start_time = datetime.now()
result = processor.process_all()
duration = (datetime.now() - start_time).total_seconds()

print(f"\n{'='*60}")
print("PROCESSING COMPLETE")
print(f"{'='*60}")
print(f"\n--- Results ---")
print(f"New papers processed:     {result['new_processed']}")
print(f"Updated papers processed: {result['updated_processed']}")
print(f"Total chunks created:     {result['total_chunks']}")
print(f"Time elapsed:             {duration:.2f} seconds")

print(f"\n--- Files Updated ---")
if settings.chunks_file.exists():
    size_mb = settings.chunks_file.stat().st_size / (1024*1024)
    print(f"✓ {settings.chunks_file} ({size_mb:.2f} MB, {result['total_chunks']} chunks)")

if settings.manifest_path.exists():
    size_kb = settings.manifest_path.stat().st_size / 1024
    papers_count = result['status']['total_papers']
    print(f"✓ {settings.manifest_path} ({size_kb:.1f} KB, {papers_count} papers)")

2026-03-20 16:50:17,671 - backend.services.paper_registry - INFO - Loaded manifest: 18 papers
2026-03-20 16:50:17,727 - backend.services.paper_processor - INFO - New: 0, Updated: 0



PROCESSING PAPERS


PROCESSING COMPLETE

--- Results ---
New papers processed:     0
Updated papers processed: 0
Total chunks created:     1387
Time elapsed:             0.99 seconds

--- Files Updated ---
✓ d:\Programming\projects\RAG\data\chunks.jsonl (2.15 MB, 1387 chunks)
✓ d:\Programming\projects\RAG\data\papers_manifest.json (3.7 KB, 18 papers)


## 4: Inspect Results


In [5]:
print("\n" + "="*60)
print("KNOWLEDGE BASE SUMMARY")
print("="*60)

# Reload registry to get fresh data from the saved manifest
registry = PaperRegistry(settings.manifest_path)
store_stats = store.get_stats()

status = registry.get_status()

print(f"\n--- Overview ---")
print(f"Papers in knowledge base:  {status['total_papers']}")
print(f"Total chunks available:    {status['total_chunks']}")
print(f"Average chunks per paper:  {status['total_chunks'] // max(1, status['total_papers']):.0f}")

if status['papers']:
    print(f"\n--- Papers Processed ---")
    for i, paper in enumerate(status['papers'], 1):
        chunks = paper['num_chunks']
        processed = paper['processed_at']
        print(f"  {i}. {paper['filename']:<40} {chunks:>4} chunks  ({processed})")

if store_stats['chunks_per_source']:
    print(f"\n--- Chunks by Source ---")
    total_checked = 0
    for source in sorted(store_stats['chunks_per_source'].keys()):
        count = store_stats['chunks_per_source'][source]
        total_checked += count
        print(f"  {Path(source).name:<40} {count:>4} chunks")
    print(f"  {'TOTAL':<40} {total_checked:>4} chunks")

2026-03-20 16:50:18,684 - backend.services.paper_registry - INFO - Loaded manifest: 18 papers



KNOWLEDGE BASE SUMMARY

--- Overview ---
Papers in knowledge base:  18
Total chunks available:    1387
Average chunks per paper:  77

--- Papers Processed ---
  1. 1906_MC_Layer_Normalization_fo.pdf         82 chunks  (2026-03-20T10:19:14.090592)
  2. accurate_uncertanities_for_dl_using_calibrated_regression.pdf   55 chunks  (2026-03-20T10:19:14.212692)
  3. Active_Learning_Framework_for_Improving_Knowledge_Graph_Accuracy.pdf   87 chunks  (2026-03-20T10:19:14.385037)
  4. automated_construction_of_theme_specific_kg.pdf   77 chunks  (2026-03-20T10:19:14.484795)
  5. dibowski-full-traceability-and-provenance-for-knowledge-graphs.pdf   63 chunks  (2026-03-20T10:19:14.585173)
  6. DOC-20251031-WA0004.pdf                    28 chunks  (2026-03-20T10:19:14.642890)
  7. DOC-20251031-WA0005.pdf                    90 chunks  (2026-03-20T10:19:14.754088)
  8. DOC-20251031-WA0006.pdf                    69 chunks  (2026-03-20T10:19:14.875557)
  9. DOC-20251031-WA0007.pdf                    98 chu

## 5: Query Chunks

Load and inspect stored chunks.

In [6]:
print(f"\nLoading chunks from {settings.chunks_file}...\n")

all_chunks = store.load_all()
print(f"✓ Loaded {len(all_chunks)} chunks")

if all_chunks:
    print(f"\n--- Sample Chunks (first 3) ---\n")
    
    for i, chunk in enumerate(all_chunks[:3], 1):
        print(f"Chunk {i}:")
        print(f"  Source:       {chunk.metadata.get('source')}")
        print(f"  Page:         {chunk.metadata.get('page')}")
        print(f"  Chunk index:  {chunk.metadata.get('chunk_index')}")
        print(f"  Size:         {len(chunk.page_content)} chars")
        print(f"  Preview:      {chunk.page_content[:80].replace(chr(10), ' ')}...")
        print()
else:
    print("No chunks found. Process papers in Step 3.")


Loading chunks from d:\Programming\projects\RAG\data\chunks.jsonl...

✓ Loaded 1387 chunks

--- Sample Chunks (first 3) ---

Chunk 1:
  Source:       d:\Programming\projects\RAG\data\raw_papers\1906_MC_Layer_Normalization_fo.pdf
  Page:         0
  Chunk index:  0
  Size:         926 chars
  Preview:      Published in Transactions on Machine Learning Research (02/2024) MC Layer Normal...

Chunk 2:
  Source:       d:\Programming\projects\RAG\data\raw_papers\1906_MC_Layer_Normalization_fo.pdf
  Page:         0
  Chunk index:  1
  Size:         987 chars
  Preview:      applications where shifts in data distribution may occur. Thus, calibrated predi...

Chunk 3:
  Source:       d:\Programming\projects\RAG\data\raw_papers\1906_MC_Layer_Normalization_fo.pdf
  Page:         0
  Chunk index:  2
  Size:         686 chars
  Preview:      and Prediction-Time Batch Normalization. Second, we explore its suitability for ...



## 6: Query by Paper

Get all chunks from a specific paper.

In [7]:
sources = store_stats['chunks_per_source'] if store_stats else {}

if sources:
    # Get first paper
    first_source = sorted(sources.keys())[0]
    paper_name = Path(first_source).name
    chunk_count = sources[first_source]
    
    print(f"\n--- Querying: {paper_name} ---")
    print(f"Expected chunks: {chunk_count}")
    
    chunks = store.get_by_source(first_source)
    
    print(f"Retrieved chunks: {len(chunks)}")
    print(f"Total size: {sum(len(c.page_content) for c in chunks)} characters")
    
    if chunks:
        print(f"\n--- Statistics ---")
        print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")
        print(f"Pages covered: {len(set(c.metadata.get('page') for c in chunks))}")
else:
    print("No sources available. Process papers first.")


--- Querying: 1906_MC_Layer_Normalization_fo.pdf ---
Expected chunks: 82
Retrieved chunks: 82
Total size: 69430 characters

--- Statistics ---
Average chunk size: 847 chars
Pages covered: 22


## 7: Detect Changes

Show what's new or modified.

In [8]:
print(f"\n--- Detecting Changes ---")
print(f"Scanning: {settings.raw_papers_dir}\n")

new_papers = registry.get_new_papers(settings.raw_papers_dir)
updated_papers = registry.get_updated_papers(settings.raw_papers_dir)

print(f"New papers (not processed):      {len(new_papers)}")
for paper in new_papers:
    print(f"  • {paper.name}")

print(f"\nModified papers (hash changed):  {len(updated_papers)}")
for paper in updated_papers:
    print(f"  • {paper.name}")

if not new_papers and not updated_papers:
    print("\n✓ Knowledge base is up to date!")


--- Detecting Changes ---
Scanning: d:\Programming\projects\RAG\data\raw_papers

New papers (not processed):      0

Modified papers (hash changed):  0

✓ Knowledge base is up to date!


## 8: Generate Embeddings

Create dense vector embeddings for all chunks using Sentence-Transformers.

In [6]:
from backend.services.embedding import EmbeddingService
from backend.services.vector_store import VectorStore

print(f"\n--- Generating Embeddings ---\n")

# Initialize embedding service
embedding_service = EmbeddingService(model_name="all-MiniLM-L6-v2")
print(f"✓ {embedding_service}\n")

# Reload chunks (in case new ones were added)
all_chunks = store.load_all()
print(f"Embedding {len(all_chunks)} chunks...")

# Generate embeddings
embedding_results = embedding_service.embed_chunks(all_chunks, batch_size=32)
print(f"✓ Generated {len(embedding_results)} embeddings")

# Initialize vector store
vector_store = VectorStore()
print(f"✓ VectorStore initialized")



2026-03-23 19:30:49,955 - backend.services.embedding - INFO - Loading embedding model: all-MiniLM-L6-v2
2026-03-23 19:30:49,962 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-03-23 19:30:49,964 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2



--- Generating Embeddings ---



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2508.53it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-23 19:30:53,728 - backend.services.embedding - INFO - Model loaded with embedding dimension: 384


✓ EmbeddingService(model_name='all-MiniLM-L6-v2', dim=384)

Embedding 1387 chunks...


2026-03-23 19:30:54,952 - backend.services.vector_store - INFO - Initialized vector store at vector_store with collection 'paper_chunks'


✓ VectorStore initialized


In [10]:
# Add embeddings to vector store
added = vector_store.add_embeddings(embedding_results, auto_delete_source=True)
print(f"✓ Added {added} embeddings\n")

# Show stats
vector_stats = vector_store.get_stats()
print(f"Vector Store Stats:")
print(f"  Total vectors: {vector_stats['total_vectors']}")
print(f"  Total sources: {vector_stats['total_sources']}")
print(f"  Embedding dimension: {embedding_service.get_embedding_dim()}")

2026-03-20 16:51:05,844 - backend.services.vector_store - INFO - Added 1387 embeddings to vector store


✓ Added 1387 embeddings

Vector Store Stats:
  Total vectors: 1387
  Total sources: 18
  Embedding dimension: 384


## 9: Semantic Search

Query the knowledge base using natural language - finds semantically similar chunks.

In [7]:
print(f"\n--- Semantic Search ---\n")

# Multiple queries to demonstrate semantic search
queries = [
    "machine learning algorithms",
    "data processing and analysis",
    "neural networks and deep learning",
    "Theme of Knowledge Graphs"
]

for query in queries:
    print(f"Query: '{query}'")
    
    # Embed the query
    query_embedding = embedding_service.embed_texts([query])[0]
    
    # Search in vector store
    results = vector_store.search(query_embedding, top_k=3, score_threshold=0.0)
    
    if results:
        print(f"  Found {len(results)} results:\n")
        for r in results:
            similarity = r['similarity_score']
            source = r['metadata'].get('source', 'unknown')
            print(f"    [ Score: {similarity:.3f} | Source: {source}")
            print(f"        {r['content']}...\n")
    else:
        print(f"  No results found\n")
    
    print()


--- Semantic Search ---

Query: 'machine learning algorithms'


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.25it/s]


  No results found


Query: 'data processing and analysis'


Batches: 100%|██████████| 1/1 [00:00<00:00, 54.13it/s]


  No results found


Query: 'neural networks and deep learning'


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.65it/s]


  No results found


Query: 'Theme of Knowledge Graphs'


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.71it/s]

  Found 3 results:

    [ Score: 0.267 | Source: d:\Programming\projects\RAG\data\raw_papers\automated_construction_of_theme_specific_kg.pdf
        Conference on Machine Learning and Knowledge Discovery in Databases, pages
20–38. Springer.
[62] Yuqi Zhu, Xiaohan Wang, Jing Chen, Shuofei Qiao, Yixin Ou, Yunzhi Yao, Shumin
Deng, Huajun Chen, and Ningyu Zhang. 2023. Llms for knowledge graph
construction and reasoning: Recent capabilities and future opportunities. arXiv
preprint arXiv:2305.13168....

    [ Score: 0.252 | Source: d:\Programming\projects\RAG\data\raw_papers\DOC-20251031-WA0008.pdf
        and 82.9% F1-macro on text-rich sources. The evaluation data and scripts used in this paper are
available in GitHub and Figshare.
Keywords Fact Veriﬁcation · Data Verbalisation · Knowledge Graphs
1
Introduction
A Knowledge Graph (KG) is a type of knowledge base that stores information in the form of semantic triples formed by
a subject, a predicate, and an object. KGs represent both real a

In [8]:
transformer = QueryTransformer(llm_model="openai/gpt-oss-20b")

q = transformer.transform(queries[-1], use_hyde=True, use_multi=True)

q

['**What is the central theme that unifies the concept of Knowledge Graphs?**',
 '**Which underlying principles or motifs define the design and purpose of Knowledge Graphs?**',
 '**How do Knowledge Graphs embody the broader theme of interconnected knowledge representation?**',
 '### The Theme of Knowledge Graphs  \n*(A comprehensive, specific exploration of what knowledge graphs represent, why they matter, and how they are built, used, and evolved.)*\n\n---\n\n## 1.  What is a Knowledge Graph?\n\n| Aspect | Description |\n|--------|-------------|\n| **Graph‑based data model** | Nodes (entities) and edges (relationships) form a network that captures real‑world facts. |\n| **Semantic enrichment** | Each node/edge is typed with a concept from an ontology, giving it meaning beyond a simple key/value pair. |\n| **Open, extensible schema** | Unlike rigid relational schemas, the graph can grow with new entities and relationships without schema migrations. |\n| **Interoperable identifiers** | 

# 11: Hybrid Search

In [12]:
from backend.services.hybrid_retriever import HybridRetriever
from backend.services.embedding import EmbeddingService

In [50]:
hybrid_retriever = HybridRetriever(vector_store, all_chunks)

query_embedding = embedding_service.embed_texts([queries[-1]])[0]

results = hybrid_retriever.search(query_embedding,queries[-1], top_k=10)
results

2026-03-23 12:32:14,057 - backend.services.hybrid_retriever - INFO - BM25 index ready with 1387 documents
Batches: 100%|██████████| 1/1 [00:00<00:00, 32.79it/s]


[{'chunk_id': 'chunk_d:\\Programming\\projects\\RAG\\data\\raw_papers\\automated_construction_of_theme_specific_kg.pdf_300',
  'content': 'Conference on Machine Learning and Knowledge Discovery in Databases, pages\n20–38. Springer.\n[62] Yuqi Zhu, Xiaohan Wang, Jing Chen, Shuofei Qiao, Yixin Ou, Yunzhi Yao, Shumin\nDeng, Huajun Chen, and Ningyu Zhang. 2023. Llms for knowledge graph\nconstruction and reasoning: Recent capabilities and future opportunities. arXiv\npreprint arXiv:2305.13168.',
  'metadata': {'creationDate': 'D:20240501001114Z',
   'modDate': 'D:20240501001114Z',
   'file_path': 'd:\\Programming\\projects\\RAG\\data\\raw_papers\\automated_construction_of_theme_specific_kg.pdf',
   'author': '',
   'subject': '-  Computing methodologies  ->  Information extraction.-  Information systems  ->  Entity relationship models.',
   'chunk_index': 76,
   'moddate': '2024-05-01T00:11:14+00:00',
   'page': 10,
   'source': 'd:\\Programming\\projects\\RAG\\data\\raw_papers\\automated_c

In [51]:
for r in results:
    print(r['metadata']['source'])

d:\Programming\projects\RAG\data\raw_papers\automated_construction_of_theme_specific_kg.pdf
d:\Programming\projects\RAG\data\raw_papers\DOC-20251031-WA0008.pdf
d:\Programming\projects\RAG\data\raw_papers\DOC-20251031-WA0010.pdf
d:\Programming\projects\RAG\data\raw_papers\DOC-20251031-WA0011.pdf
d:\Programming\projects\RAG\data\raw_papers\automated_construction_of_theme_specific_kg.pdf
d:\Programming\projects\RAG\data\raw_papers\automated_construction_of_theme_specific_kg.pdf
d:\Programming\projects\RAG\data\raw_papers\expanding_kg_with_human_in_loop.pdf
d:\Programming\projects\RAG\data\raw_papers\DOC-20251031-WA0010.pdf
d:\Programming\projects\RAG\data\raw_papers\automated_construction_of_theme_specific_kg.pdf
d:\Programming\projects\RAG\data\raw_papers\DOC-20251031-WA0010.pdf


# 12: ReRanking

In [9]:
import importlib

In [10]:
from backend.services.reranker import CrossEncoderReRanker

In [11]:
importlib.reload(importlib.import_module("backend.services.reranker"))

<module 'backend.services.reranker' from 'd:\\Programming\\projects\\RAG\\backend\\services\\reranker.py'>

In [12]:
reranker = CrossEncoderReRanker()
final = reranker.rerank(queries[-1], results, top_k=5)

final

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1638.80it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-23 19:31:19,176 - sentence_transformers.cross_encoder.CrossEncoder - INFO - Use pytorch device: cpu
2026-03-23 19:31:19,632 - backend.services.reranker - INFO - Loading cross-encoder model: CrossEncoder(
  (model): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 384, padding_idx=0)
        (position_embeddings): Embedding(512, 384)
        (token_type_embeddings): Embedding(2, 384)
        (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(

[{'chunk_id': 'chunk_d:\\Programming\\projects\\RAG\\data\\raw_papers\\DOC-20251031-WA0008.pdf_651',
  'content': 'and 82.9% F1-macro on text-rich sources. The evaluation data and scripts used in this paper are\navailable in GitHub and Figshare.\nKeywords Fact Veriﬁcation · Data Verbalisation · Knowledge Graphs\n1\nIntroduction\nA Knowledge Graph (KG) is a type of knowledge base that stores information in the form of semantic triples formed by\na subject, a predicate, and an object. KGs represent both real and abstract entities internally as labelled and uniquely\nidentiﬁable entities, such as The Moon or Happiness, and can amass information from a multitude of domains and\nsources by connecting such entities amongst themselves or to literals through relationships, coded via uniquely iden-\ntiﬁed predicates. KGs serve as sources of both human and machine-readable semantically structured data for various\ncrucial applications in the modern web landscape, such as Wikipedia infoboxes, sea

In [13]:
# lets print scores and check 
for r in results: 
    print(r['chunk_id'][-3:])
    print(r['metadata']['source'], r['similarity_score'], r['combined_score'],)

300


KeyError: 'combined_score'

In [ ]:
for r in final: 
    print(r['chunk_id'][-3:])
    print(r['metadata']['source'], r['similarity_score'], r['combined_score'], r['rerank_score'])

224
d:\Programming\projects\RAG\data\raw_papers\automated_construction_of_theme_specific_kg.pdf 0.23735570907592773 0.009230769230769232 7.245584487915039
227
d:\Programming\projects\RAG\data\raw_papers\automated_construction_of_theme_specific_kg.pdf 0.22075140476226807 0.008823529411764706 6.596659183502197
937
d:\Programming\projects\RAG\data\raw_papers\DOC-20251031-WA0011.pdf 0.24586844444274902 0.009523809523809523 3.7907626628875732
651
d:\Programming\projects\RAG\data\raw_papers\DOC-20251031-WA0008.pdf 0.2517472505569458 0.009836065573770493 3.546529769897461
927
d:\Programming\projects\RAG\data\raw_papers\DOC-20251031-WA0010.pdf 0.2201220989227295 0.008695652173913044 3.069868803024292


# 13: Generating Answers

In [ ]:
from backend.services.generation import GroqGenerator

In [ ]:
importlib.reload(importlib.import_module("backend.services.generation"))

<module 'backend.services.generation' from 'd:\\Programming\\projects\\RAG\\backend\\services\\generation.py'>

In [ ]:
generator = GroqGenerator()


2026-03-23 18:25:29,816 - backend.services.generation - INFO - Loading Groq model: openai/gpt-oss-20b
2026-03-23 18:25:29,872 - backend.services.generation - INFO - Groq model loaded: openai/gpt-oss-20b


In [ ]:
result = generator.generate_with_metadata(queries[-1], final)
for r in result: 
    print(r)
    print(result[r])

answer
Knowledge graphs (KGs) are directed, labeled multigraphs that encode real‑world knowledge as triples ⟨h, r, t⟩ linking entities (h, t) through relations (r) [3].  They are used across a wide range of applications—including recommendation systems, question answering, intelligent conversational systems, and medical concept modeling—by providing structured, semantically rich data that can be queried and reasoned over [2], [4].  KGs can be **generic open‑world** (e.g., Wikidata) or **domain‑specific** (e.g., UMLS), but both types face challenges such as limited granularity and lack of timeliness.  To address these issues, theme‑specific KGs are being constructed automatically to capture fine‑grained, up‑to‑date knowledge for specialized domains or rapidly evolving contexts [1].
sources
[{'source': 'd:\\Programming\\projects\\RAG\\data\\raw_papers\\automated_construction_of_theme_specific_kg.pdf', 'page': 0, 'rerank_score': None, 'preview': 'Automated Construction of Theme-specific K

# 14: Using presets

In [14]:
from backend.services.preset_registry import PresetRegistry

# Initialize PresetRegistry
preset_registry = PresetRegistry()

print(f"Available presets: {preset_registry.list_presets()}")

2026-03-23 19:31:21,087 - backend.services.preset_registry - INFO - Loaded preset: qa
2026-03-23 19:31:21,094 - backend.services.preset_registry - INFO - Loaded preset: research
2026-03-23 19:31:21,101 - backend.services.preset_registry - INFO - Loaded preset: simple


Available presets: ['qa', 'research', 'simple']


In [45]:

config_module = importlib.reload(importlib.import_module("backend.core.config"))

Settings = config_module.Settings
settings = config_module.settings



In [31]:
from backend.services.rag_pipeline import RAGPipeline

In [32]:
importlib.reload(importlib.import_module("backend.services.rag_pipeline"))

<module 'backend.services.rag_pipeline' from 'd:\\Programming\\projects\\RAG\\backend\\services\\rag_pipeline.py'>

In [33]:
rag = RAGPipeline(
    chunk_store_path=str(settings.chunks_file),
    vector_store_path= str(settings.vector_store_dir),
    preset_registry=preset_registry
)


2026-03-23 19:35:14,071 - backend.services.embedding - INFO - Loading embedding model: all-MiniLM-L6-v2
2026-03-23 19:35:14,079 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-03-23 19:35:14,081 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2172.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-23 19:35:17,792 - backend.services.embedding - INFO - Model loaded with embedding dimension: 384
2026-03-23 19:35:17,820 - backend.services.vector_store - INFO - Initialized vector store at d:\Programming\projects\RAG\data\vector_store with collection 'paper_chunks

In [34]:
query = "What our Knowledge Graphs?"
result = rag.query(query, preset="simple")

print(f"Query: {result['query']}")
print(f"{'='*60}")
print(f"\nAnswer:\n{result['answer']}\n")
print(f"Sources: {result['num_sources']}")
print(f"Preset Used: {result['metadata']['preset_used']}")
print(f"Timing: {result['timing_ms']:.2f}ms")

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s]


Query: What our Knowledge Graphs?

Answer:
**Answer:**  
A knowledge graph is a structured collection of facts that links entities (like people, places, or concepts) with relationships between them. It represents information as a network of nodes (entities) and edges (relationships), allowing computers to understand and reason about the connections in the data.

Sources: 1
Preset Used: simple
Timing: 1087.57ms


In [36]:
result = rag.query(query, preset="qa")


print(f"Query: {result['query']}")
print(f"{'='*60}")
print(f"\nAnswer:\n{result['answer']}\n")
print(f"Sources: {result['num_sources']}")
print(f"Preset Used: {result['metadata']['preset_used']}")
print(f"Timing: {result['timing_ms']:.2f}ms")

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.20it/s]


Query: What our Knowledge Graphs?

Answer:
**What are our Knowledge Graphs?**

Knowledge Graphs (KGs) are structured, graph‑based representations of real‑world facts.  
They consist of **entities** (nodes), **relations** (edges), and **triples** that link an entity to another via a relation. Formally a KG can be described as a directed, labeled multigraph \((E, R, T)\) where  

* \(E\) – set of entities (e.g., *Barack Obama*, *Amazon*, *UMLS* concepts)  
* \(R\) – set of relation types (e.g., *instanceOf*, *locatedIn*, *offers*)  
* \(T\) – set of triples \(\langle h, r, t\rangle \in E \times R \times E\) (e.g., \(\langle\)Barack Obama, instanceOf, Person\(\rangle\))  

These triples encode factual knowledge that can be queried, reasoned over, and used to power downstream AI systems.

---

### 1. Why KGs matter

* **Machine‑readable knowledge** – KGs translate human domain knowledge into a format that algorithms can process, enabling generalisation beyond the data a model was trained o

# 15: Trying custom preset

In [38]:
from backend.services.rag_pipeline import RAGConfig

In [43]:
custom_preset = {
    "retrieval": {
        "strategy": "dense",              
        "top_k": 6,
        "dense_weight": 1,
        "query_expansion": False,
    },
    "reranking": {
        "enabled": False,                 
        "top_k": 3,
        "threshold": 0.0,
    },
    "generation": {
        "temperature": 0.1,               
        "response_format": "concise",
        "system_message": "You are a precise expert. Answer in 2-3 sentences maximum using only the provided sources.",
        "include_reasoning": False,
    }
}

custom_config = RAGConfig.from_preset(custom_preset)

In [44]:
result = rag.query(query, config=custom_config)

result

Batches: 100%|██████████| 1/1 [00:00<00:00, 43.48it/s]


{'query': 'What our Knowledge Graphs?',
 'answer': 'Knowledge graphs are structured, graph‑based representations of real‑world entities and their interrelationships, enabling machines to understand context and semantics. They combine data from multiple sources into a unified schema, allowing advanced search, recommendation, and AI inference.',
 'sources': [],
 'num_sources': 0,
 'metadata': {'preset_used': None,
  'retrieval_strategy': 'dense',
  'reranking_enabled': False,
  'query_expansion': False,
  'num_queries_expanded': 1,
  'num_candidates_retrieved': 0},
 'timing_ms': 307.62696266174316}